In [0]:
# ================================
# Test Spark
# ================================
print(f"Spark version : {spark.version}")
print("Spark fonctionne !")

In [0]:
# Charger le fichier WBAN 
df_wban = spark.read.csv(
    "/Volumes/workspace/default/airport_time_zone/wban_airport_timezone.csv",
    header=True,
    inferSchema=True
)

print(f"📊 Lignes WBAN  : {df_wban.count():,}")
print(f"📋 Colonnes     : {df_wban.columns}")
df_wban.show(10, truncate=False)

In [0]:
# Voir les fichiers disponibles
display(dbutils.fs.ls("/Volumes/workspace/default/flights/"))

In [0]:
from pyspark.sql.functions import col, count, when, round as spark_round

df_aotp = spark.read.csv(
    "/Volumes/workspace/default/flights/",
    header=True,
    inferSchema=True,
    nullValue="NA"
)

print(f"📊 Total vols bruts     : {df_aotp.count():,}")
print(f"📋 Colonnes             : {df_aotp.columns}")
df_aotp.show(3)

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/weather/"))

In [0]:
from pyspark.sql.functions import col, count, when, round as spark_round

df_weather = spark.read.csv(
    "/Volumes/workspace/default/weather/",
    header=True,
    inferSchema=True,
    nullValue="M"
)

print(f"📊 Total observations météo : {df_weather.count():,}")
print(f"📋 Colonnes                 : {df_weather.columns}")
df_weather.show(3)

In [0]:
# Reproduire Table III du papier
df_aotp.groupBy("FL_DATE").agg(
    count("*").alias("Total_Vols"),
    spark_round(
        count(when(
            (col("CANCELLED")==0) &
            (col("DIVERTED")==0) &
            (col("ARR_DELAY_NEW") < 15), 1
        )) / count("*") * 100, 1
    ).alias("% OnTime"),
    spark_round(
        count(when(
            (col("CANCELLED")==0) &
            (col("DIVERTED")==0) &
            (col("ARR_DELAY_NEW") >= 15), 1
        )) / count("*") * 100, 1
    ).alias("% Delayed"),
    spark_round(
        count(when(col("CANCELLED")==1, 1)
    ) / count("*") * 100, 1
    ).alias("% Cancelled"),
    spark_round(
        count(when(col("DIVERTED")==1, 1)
    ) / count("*") * 100, 1
    ).alias("% Diverted")
).orderBy("FL_DATE").show()

In [0]:
# =========================================
# TABLE III DU PAPIER — Stats par année ✅
# Résultats validés vs papier :
# 2012 : 81.9% ontime ✅ | 16.7% delayed ✅
# 2013 : 78.3% ontime ✅ | 19.9% delayed ✅
# =========================================
from pyspark.sql.functions import year, col, count, when
from pyspark.sql.functions import round as spark_round

df_stats_year = df_aotp.groupBy(
    year(col("FL_DATE")).alias("Year")
).agg(
    count("*").alias("Total_Vols"),
    spark_round(
        count(when(
            (col("CANCELLED") == 0) &
            (col("DIVERTED")  == 0) &
            (col("ARR_DELAY_NEW").isNotNull()) &
            (col("ARR_DELAY_NEW") < 15), 1
        )) / count("*") * 100, 1
    ).alias("% OnTime"),
    spark_round(
        count(when(
            (col("CANCELLED") == 0) &
            (col("DIVERTED")  == 0) &
            (col("ARR_DELAY_NEW").isNotNull()) &
            (col("ARR_DELAY_NEW") >= 15), 1
        )) / count("*") * 100, 1
    ).alias("% Delayed"),
    spark_round(
        count(when(col("CANCELLED") == 1, 1)
    ) / count("*") * 100, 1
    ).alias("% Cancelled"),
    spark_round(
        count(when(col("DIVERTED") == 1, 1)
    ) / count("*") * 100, 1
    ).alias("% Diverted")
).orderBy("Year")

# Afficher et sauvegarder
df_stats_year.show()

In [0]:
# Créer le volume outputs dans Databricks
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.outputs")
print("✅ Volume outputs créé !")

In [0]:
# Sauvegarder Table III dans le volume outputs
df_stats_year.write.mode("overwrite").csv(
    "/Volumes/workspace/default/outputs/stats_year/",
    header=True
)
print("✅ Table III sauvegardée !")

In [0]:
# COMMAND ----------
# ================================
# ✅ NOUVEAU — TABLE IV du papier
# Causes de retard par année
# Section 3 du papier
# ================================
df_delayed = df_aotp.filter(
    (col("CANCELLED") == 0) &
    (col("DIVERTED")  == 0) &
    col("ARR_DELAY_NEW").isNotNull() &
    (col("ARR_DELAY_NEW") >= 15)
)

df_table4 = df_delayed.groupBy(
    year(col("FL_DATE")).alias("Year")
).agg(
    count("*").alias("Total_Delayed"),

    # % Weather Delay (Extreme Weather)
    spark_round(
        count(when(col("WEATHER_DELAY") > 0, 1)) /
        count("*") * 100, 1
    ).alias("% ExtremeWeather"),

    # % NAS Delay
    spark_round(
        count(when(col("NAS_DELAY") > 0, 1)) /
        count("*") * 100, 1
    ).alias("% NAS")
).orderBy("Year")

print("=== TABLE IV DU PAPIER — Causes de retard ===")
print("(limité à WeatherDelay et NAS car autres colonnes absentes)")
df_table4.show()

# COMMAND ----------
# ================================
# ✅ NOUVEAU — TABLE V du papier
# Retards liés à la météo par année
# Section 3 du papier
# ================================
df_table5 = df_delayed.groupBy(
    year(col("FL_DATE")).alias("Year")
).agg(
    count("*").alias("Total_Delayed"),

    # % retards Extreme Weather
    spark_round(
        count(when(
            (col("WEATHER_DELAY") > 0) &
            (col("NAS_DELAY") == 0), 1
        )) / count("*") * 100, 1
    ).alias("% ExtremeWeather_only"),

    # % retards NAS liés météo
    spark_round(
        count(when(
            (col("NAS_DELAY") > 0) &
            (col("WEATHER_DELAY") == 0), 1
        )) / count("*") * 100, 1
    ).alias("% NAS_only"),

    # % retards combinés Weather + NAS
    spark_round(
        count(when(
            (col("WEATHER_DELAY") > 0) &
            (col("NAS_DELAY") > 0), 1
        )) / count("*") * 100, 1
    ).alias("% Weather_AND_NAS"),

    # % total météo = Weather + NAS
    spark_round(
        count(when(
            (col("WEATHER_DELAY") > 0) |
            (col("NAS_DELAY") > 0), 1
        )) / count("*") * 100, 1
    ).alias("% Total_Weather")
).orderBy("Year")

print("=== TABLE V DU PAPIER — Retards météo ===")
df_table5.show()

In [0]:
df_table4.write.mode("overwrite").csv(
    "/Volumes/workspace/default/outputs/table4/",
    header=True
)
print("✅ Table IV sauvegardée !")

df_table5.write.mode("overwrite").csv(
    "/Volumes/workspace/default/outputs/table5/",
    header=True
)
print("✅ Table V sauvegardée !")

In [0]:
# =========================================
# Section 4.1 du papier :
# "filtered out diverted and canceled 
#  flights → Flight Table FT"
# =========================================
from pyspark.sql.functions import (
    col, expr, floor, to_timestamp, 
    lpad, concat, lit
)

df_FT = df_aotp.filter(
    (col("CANCELLED") == 0) &
    (col("DIVERTED")  == 0) &
    (col("ARR_DELAY_NEW").isNotNull())
).select(
    "FL_DATE",
    "ORIGIN_AIRPORT_ID",
    "DEST_AIRPORT_ID",
    "CRS_DEP_TIME",
    "CRS_ELAPSED_TIME",
    "ARR_DELAY_NEW",
    "WEATHER_DELAY",
    "NAS_DELAY"
)

# Remplir delays manquants par 0
df_FT = df_FT.fillna(0, subset=["WEATHER_DELAY", "NAS_DELAY"])

# ✅ CORRECTION CRS_ARR_TIME
# Convertir HHMM → minutes → ajouter durée → reconvertir HHMM
# Ex: 900 → 540min + 77min = 617min → 10h17 → 1017 ✅
df_FT = df_FT.withColumn(
    "dep_min",
    (floor(col("CRS_DEP_TIME") / 100) * 60 +
     col("CRS_DEP_TIME") % 100)
).withColumn(
    "arr_min",
    col("dep_min") + col("CRS_ELAPSED_TIME")
).withColumn(
    # Modulo 1440 pour gérer les vols après minuit
    "CRS_ARR_TIME",
    (floor((col("arr_min") % 1440) / 60) * 100 +
     (col("arr_min") % 1440) % 60).cast("integer")
).drop("dep_min", "arr_min")

# ✅ Forcer types integer pour jointure
df_FT = df_FT.withColumn(
    "ORIGIN_AIRPORT_ID",
    col("ORIGIN_AIRPORT_ID").cast("integer")
).withColumn(
    "DEST_AIRPORT_ID",
    col("DEST_AIRPORT_ID").cast("integer")
)

# ✅ NOUVEAU — DEP_DATETIME pour le join temporel
# Le papier dit : join key = (airport, DATE(t_sd))
df_FT = df_FT.withColumn(
    "DEP_DATETIME",
    to_timestamp(
        concat(
            col("FL_DATE"),
            lit(" "),
            lpad(col("CRS_DEP_TIME").cast("string"), 4, "0")
        ),
        "yyyy-MM-dd HHmm"
    )
).withColumn(
    # ✅ NOUVEAU — ARR_DATETIME pour le 2ème join (météo destination)
    "ARR_DATETIME",
    to_timestamp(
        concat(
            col("FL_DATE"),
            lit(" "),
            lpad(col("CRS_ARR_TIME").cast("string"), 4, "0")
        ),
        "yyyy-MM-dd HHmm"
    )
)

# Vérification CRS_ARR_TIME
print("=== Vérification CRS_ARR_TIME ===")
df_FT.select(
    "CRS_DEP_TIME",
    "CRS_ELAPSED_TIME",
    "CRS_ARR_TIME",
    "DEP_DATETIME",
    "ARR_DATETIME"
).show(5)
# Résultat attendu :
# DEP=900  + 77min  → ARR=1017  ✅
# DEP=1040 + 79min  → ARR=1159  ✅
# DEP=1227 + 175min → ARR=1502  ✅

print(f"✅ Flight Table FT : {df_FT.count():,} vols")
print(f"📋 Colonnes        : {df_FT.columns}")
df_FT.show(5)

# Sauvegarder FT
df_FT.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/outputs/FT/"
)
print("✅ FT sauvegardée → /Volumes/workspace/default/outputs/FT/")

In [0]:
from pyspark.sql.functions import col, lpad, concat, to_timestamp, lit, expr

# Forcer types pour jointure
df_wban_clean = df_wban.withColumn(
    "WBAN", col("WBAN").cast("integer")
).withColumn(
    "AirportID", col("AirportID").cast("integer")
)

df_weather_clean = df_weather.withColumn(
    "WBAN", col("WBAN").cast("integer")
)

# Joindre weather avec WBAN → avoir AirportID
df_OT = df_weather_clean.join(
    df_wban_clean, "WBAN", "inner"
).select(
    col("AirportID"),
    col("WBAN"),
    col("Date"),
    col("Time"),
    # ✅ try_cast pour tolérer valeurs malformées → NULL si invalide
    expr("try_cast(DryBulbCelsius as double)").alias("Temp"),
    expr("try_cast(RelativeHumidity as double)").alias("Humidity"),
    expr("try_cast(WindDirection as double)").alias("WindDirection"),
    expr("try_cast(WindSpeed as double)").alias("WindSpeed"),
    expr("try_cast(StationPressure as double)").alias("Pressure"),
    col("SkyCondition"),
    expr("try_cast(Visibility as double)").alias("Visibility"),
    col("WeatherType"),
    col("TimeZone").cast("integer")
)

# Créer colonne datetime unifiée
df_OT = df_OT.withColumn(
    "OBS_DATETIME",
    to_timestamp(
        concat(
            col("Date"),
            lit(" "),
            lpad(col("Time"), 4, "0")
        ),
        "yyyyMMdd HHmm"
    )
)

# Nettoyer valeurs manquantes critiques
df_OT = df_OT.dropna(
    subset=["AirportID", "Date", "Time", "OBS_DATETIME"]
)

print(f"✅ Weather Table OT : {df_OT.count():,} observations")
print(f"📋 Colonnes         : {df_OT.columns}")
df_OT.show(5)

# Vérifier combien de NULL après try_cast
print("\n=== Valeurs NULL après try_cast ===")
from pyspark.sql.functions import count, when, isnan
df_OT.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in ["Temp","Humidity","WindDirection",
              "WindSpeed","Pressure","Visibility"]
]).show()

# Sauvegarder OT
df_OT.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/outputs/OT/"
)
print("✅ OT sauvegardée → /Volumes/workspace/default/outputs/OT/")

In [0]:
# Vérification croisée aéroports
airports_FT = df_FT.select(
    col("ORIGIN_AIRPORT_ID").alias("AirportID")
).union(
    df_FT.select(
        col("DEST_AIRPORT_ID").alias("AirportID")
    )
).distinct()

airports_OT = df_OT.select("AirportID").distinct()

n_commun = airports_FT.join(
    airports_OT, "AirportID", "inner"
).count()

print("=" * 45)
print("      RÉSUMÉ FINAL — NOTEBOOK 01          ")
print("=" * 45)
print(f"✈️  Flight Table  FT : {df_FT.count():,} vols")
print(f"🌤️  Weather Table OT : {df_OT.count():,} observations")

if n_commun == 0:
    print("⚠️ PROBLÈME CRITIQUE : 0 aéroports communs !")
    print("Vérification types...")
    print("FT type :", df_FT.schema["ORIGIN_AIRPORT_ID"].dataType)
    print("OT type :", df_OT.schema["AirportID"].dataType)
    print("Exemples FT :")
    df_FT.select("ORIGIN_AIRPORT_ID").show(3)
    print("Exemples OT :")
    df_OT.select("AirportID").show(3)
else:
    print(f"print(f"🔗 Aéroports communs : {n_commun}") → Join possible !")
    
print("-" * 45)
print("Colonnes FT :", df_FT.columns)
print("Colonnes OT :", df_OT.columns)
print("=" * 45)
print("✅ Notebook 01 TERMINÉ → Prêt pour Notebook 02 !")